In [3]:
import pandas as pd
import numpy as np

# ==========================================
# 1. LOAD DATA
# ==========================================
def load_nsl():
    cols = ["duration","protocol_type","service","flag","src_bytes",
        "dst_bytes","land","wrong_fragment","urgent","hot","num_failed_logins",
        "logged_in","num_compromised","root_shell","su_attempted","num_root",
        "num_file_creations","num_shells","num_access_files","num_outbound_cmds",
        "is_host_login","is_guest_login","count","srv_count","serror_rate",
        "srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
        "diff_srv_rate","srv_diff_host_rate","dst_host_count","dst_host_srv_count",
        "dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate",
        "dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate",
        "dst_host_rerror_rate","dst_host_srv_rerror_rate","label", "difficulty"]
    
    df = pd.read_csv('KDDTrain+.txt', names=cols)
    test = pd.read_csv('KDDTest+.txt', names=cols)
    df = pd.concat([df, test])
    
    # Label: Normal=0, Attack=1
    df['label'] = df['label'].apply(lambda x: 0 if x == 'normal' else 1)
    return df

def load_unsw():
    df = pd.read_csv('UNSW_NB15_training-set.csv')
    test = pd.read_csv('UNSW_NB15_testing-set.csv')
    df = pd.concat([df, test])
    return df

# ==========================================
# 2. AGGRESSIVE MAPPING DICTIONARY
# ==========================================
# We will map NSL-KDD names TO UNSW-NB15 names
mapping_nsl_to_unsw = {
    # Base Network Flow
    'duration': 'dur',
    'protocol_type': 'proto',
    'service': 'service',
    'src_bytes': 'sbytes',
    'dst_bytes': 'dbytes',
    'land': 'is_sm_ips_ports',
    
    # Content / Flags
    'wrong_fragment': 'trans_depth', # (Loose map: fragmentation implies depth issues)
    'is_guest_login': 'is_ftp_login', # (Partial overlap)
    
    # Time-Based Traffic Features (The "Count" variables)
    # NSL 'count' (2s window) ~~ UNSW 'ct_dst_src_ltm' (dest-src count)
    'count': 'ct_dst_src_ltm', 
    
    # NSL 'srv_count' (2s window) ~~ UNSW 'ct_srv_dst' (service-dest count)
    'srv_count': 'ct_srv_dst',

    # NSL 'dst_host_count' (100 connection window) ~~ UNSW 'ct_dst_ltm'
    'dst_host_count': 'ct_dst_ltm',
    
    # NSL 'dst_host_srv_count' ~~ UNSW 'ct_src_ltm'
    'dst_host_srv_count': 'ct_src_ltm',
    
    # NSL 'dst_host_same_src_port_rate' ~~ UNSW 'ct_dst_sport_ltm'
    # (We map rate 0-1 to count 0-N by just accepting they are correlated)
    'dst_host_same_src_port_rate': 'ct_dst_sport_ltm',
}

# ==========================================
# 3. APPLY MAPPING & MERGE
# ==========================================
df_nsl = load_nsl()
df_unsw = load_unsw()

print(f"Original NSL Shape: {df_nsl.shape}")
print(f"Original UNSW Shape: {df_unsw.shape}")

# 1. Rename NSL columns to match UNSW
# We keep only the columns present in the mapping + label
nsl_mapped = df_nsl.rename(columns=mapping_nsl_to_unsw)
common_cols = list(mapping_nsl_to_unsw.values()) + ['label']

# 2. Filter both datasets to only these columns
# (Check if any keys from mapping are missing in UNSW)
final_cols = [c for c in common_cols if c in df_unsw.columns]

print(f"\nMerging on {len(final_cols)} common features:")
print(final_cols)

df_nsl_final = nsl_mapped[final_cols].copy()
df_unsw_final = df_unsw[final_cols].copy()

# 3. Clean Categorical Data
# Protocol
df_nsl_final['proto'] = df_nsl_final['proto'].str.lower()
df_unsw_final['proto'] = df_unsw_final['proto'].str.lower()

# Service
df_nsl_final['service'] = df_nsl_final['service'].str.lower()
df_unsw_final['service'] = df_unsw_final['service'].str.lower()
df_unsw_final['service'].replace('-', 'unknown', inplace=True)

df_nsl_final['dataset'] = 'NSL-KDD'
df_unsw_final['dataset'] = 'UNSW-NB15'

# 5. Concatenate
combined_df = pd.concat([df_nsl_final, df_unsw_final], axis=0, ignore_index=True)

print(f"\nFinal Combined Shape: {combined_df.shape}")
combined_df.to_csv("Merged_IDS.csv", index=False)
print("Saved to Merged_IDS.csv")

Original NSL Shape: (148517, 43)
Original UNSW Shape: (257673, 45)

Merging on 14 common features:
['dur', 'proto', 'service', 'sbytes', 'dbytes', 'is_sm_ips_ports', 'trans_depth', 'is_ftp_login', 'ct_dst_src_ltm', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm', 'ct_dst_sport_ltm', 'label']


C:\Users\Administrator\AppData\Local\Temp\ipykernel_10632\840569628.py:100: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_unsw_final['service'].replace('-', 'unknown', inplace=True)



Final Combined Shape: (406190, 15)
Saved to Merged_IDS.csv
